# Ранняя аналитика: top-10 only_excel INN в clients/agreements

Что делает тетрадка:
1. Формирует `only_excel` INN (Excel minus merchants snapshot на 1-е число месяца).
2. Берет top-10 этих INN.
3. Проверяет наличие INN в `ods_alpha.scd1_companies`.
4. Проверяет наличие договоров в `ods_alpha.scd1_agreements`.
5. Показывает типы договоров (`acq_class`) и детали для сравнения с Excel:
   - INN
   - Наименование клиента
   - Номер договора
   - ID договора
   - Тип договора


In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = "/home/jovyan/documents/Equaring/Проверки/отчет_февраль_2026.xlsx"
excel_inn_col = "ИНН"
excel_contract_col = "Номер договора"
excel_name_col = "Наименование клиента"  # если в Excel другое имя, поменяйте здесь

month_start = "2026-02-01"
top_n_only_excel_inn = 10

merchants_snapshot_table = "sandbox_ai.stg__chesnov_aef__sa_acquiring_merchants"

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
def normalize_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    return s if re.fullmatch(r"\d+", s) else None

def normalize_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s

def to_sql_in_list(values):
    return ','.join([f"'{v}'" for v in values])

## 1) Загружаем Excel и формируем список INN

In [ ]:
df_excel = pd.read_excel(excel_path)
if excel_inn_col not in df_excel.columns:
    raise ValueError(f"В Excel не найдена колонка: {excel_inn_col}")

excel_norm = pd.DataFrame({
    'inn': df_excel[excel_inn_col].apply(normalize_id),
    'excel_contract_number': df_excel[excel_contract_col].apply(normalize_str) if excel_contract_col in df_excel.columns else None,
    'excel_client_name': df_excel[excel_name_col].apply(normalize_str) if excel_name_col in df_excel.columns else None,
})
excel_norm = excel_norm.dropna(subset=['inn']).copy()

excel_inn_set = set(excel_norm['inn'].dropna().tolist())
print('excel_unique_inn =', len(excel_inn_set))
display(excel_norm.head(20))

## 2) INN из merchants snapshot (активные на 1-е число месяца)

In [ ]:
with imp:
    dm_inn = imp.fetch(f"""
        select distinct cast(inn as string) as inn
        from {merchants_snapshot_table}
        where cast(d_valid_from as date) <= cast('{month_start}' as date)
          and (
                d_valid_to is null
                or cast(d_valid_to as date) >= cast('{month_start}' as date)
              )
          and inn is not null
    """)

dm_inn_set = set(dm_inn['inn'].apply(normalize_id).dropna().tolist())
print('merchants_unique_inn_active_on_1st =', len(dm_inn_set))

## 3) Формируем top-10 only_excel INN

In [ ]:
only_excel_inn = sorted(list(excel_inn_set - dm_inn_set))
top10_only_excel_inn = only_excel_inn[:top_n_only_excel_inn]

print('only_excel_inn_total =', len(only_excel_inn))
print('top10_selected =', len(top10_only_excel_inn))
display(pd.DataFrame({'inn': top10_only_excel_inn}))

## 4) Проверяем наличие top-10 INN в clients (`ods_alpha.scd1_companies`)

In [ ]:
if len(top10_only_excel_inn) == 0:
    clients_presence = pd.DataFrame(columns=['inn', 'n_cmp', 'c_cmp_name'])
else:
    in_list = to_sql_in_list(top10_only_excel_inn)
    with imp:
        clients_presence = imp.fetch(f"""
            select
                cast(c_inn as string) as inn,
                cast(n_cmp as string) as n_cmp,
                cast(c_cmp_name as string) as c_cmp_name
            from ods_alpha.scd1_companies
            where cast(c_inn as string) in ({in_list})
        """)

clients_presence = clients_presence.drop_duplicates()
display(clients_presence)

clients_presence_set = set(clients_presence['inn'].dropna().tolist())
clients_stage = pd.DataFrame({'inn': top10_only_excel_inn})
clients_stage['found_in_clients'] = clients_stage['inn'].isin(clients_presence_set)
display(clients_stage)

## 5) Проверяем договоры этих клиентов и типы договоров (`agreements.acq_class`)

In [ ]:
if len(top10_only_excel_inn) == 0:
    agreements_details = pd.DataFrame(columns=['inn','client_name','agr_id','c_agr_number','acq_class','d_valid_from','d_valid_to'])
else:
    in_list = to_sql_in_list(top10_only_excel_inn)
    with imp:
        agreements_details = imp.fetch(f"""
            select
                cast(c.c_inn as string) as inn,
                cast(c.c_cmp_name as string) as client_name,
                cast(a.abs_agr_id as string) as agr_id,
                cast(a.c_agr_number as string) as c_agr_number,
                cast(a.acq_class as string) as acq_class,
                cast(a.d_valid_from as date) as d_valid_from,
                cast(a.d_valid_to as date) as d_valid_to
            from ods_alpha.scd1_companies c
            left join ods_alpha.scd1_agreements a
              on a.n_cmp_client = c.n_cmp
            where cast(c.c_inn as string) in ({in_list})
        """)

agreements_details = agreements_details.sort_values(['inn','c_agr_number','agr_id'])
display(agreements_details.head(200))

## 6) Сводка по этапам для top-10 INN

In [ ]:
stage = pd.DataFrame({'inn': top10_only_excel_inn})

if len(clients_presence) > 0:
    stage['found_in_clients'] = stage['inn'].isin(set(clients_presence['inn'].dropna().tolist()))
else:
    stage['found_in_clients'] = False

if len(agreements_details) > 0:
    agr_cnt = agreements_details.groupby('inn', as_index=False)['agr_id'].nunique().rename(columns={'agr_id': 'agreements_cnt'})
    sa_cnt = agreements_details[agreements_details['acq_class'] == 'SA'].groupby('inn', as_index=False)['agr_id'].nunique().rename(columns={'agr_id': 'sa_agreements_cnt'})
    stage = stage.merge(agr_cnt, on='inn', how='left')
    stage = stage.merge(sa_cnt, on='inn', how='left')
else:
    stage['agreements_cnt'] = 0
    stage['sa_agreements_cnt'] = 0

stage['agreements_cnt'] = stage['agreements_cnt'].fillna(0).astype(int)
stage['sa_agreements_cnt'] = stage['sa_agreements_cnt'].fillna(0).astype(int)
stage['has_any_agreement'] = stage['agreements_cnt'] > 0
stage['has_sa_agreement'] = stage['sa_agreements_cnt'] > 0

display(stage)

## 7) Данные для сравнения с Excel (ИНН + наименование + номер договора)

Слева — данные из Озера, справа — строки из Excel по тем же INN.

In [ ]:
excel_cmp = excel_norm[excel_norm['inn'].isin(top10_only_excel_inn)].copy()
excel_cmp = excel_cmp[['inn','excel_client_name','excel_contract_number']].drop_duplicates()

lake_cmp = agreements_details[['inn','client_name','c_agr_number','agr_id','acq_class','d_valid_from','d_valid_to']].drop_duplicates()

display(lake_cmp.head(200))
display(excel_cmp.head(200))